[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Cyclostationary_HOS.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Cyclostationarity & Higher-Order Statistics

Two escapes from the [WSS/Gaussian](./Statistical_Signal_Processing.ipynb) worldview: signals whose *statistics* repeat periodically (every modulated signal!), detectable even **below the noise floor** — and higher-order moments that see what covariance is blind to. This is how [SDR](../Intro_SDR/Software_Defined_Radio.ipynb) detectors find signals the PSD can't.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb), [Digital Communications](./Digital_Communications.ipynb) S1–S2.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Cyclostationarity: Rhythm in the Statistics* (~40 min)
**Goal:** see why modulated signals aren't WSS; meet the cyclic autocorrelation.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S1. &nbsp; **Feeds into:** Session 2 (detection below the noise floor).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Cyclostationarity — Rhythm in the Statistics</b></summary>

**Timing (~40 min).** 10 min why modulated signals are not WSS · 10 min the cyclic autocorrelation · 12 min the demo and its spike locations · 8 min the asymmetry that makes it useful.

**Board first — break WSS concretely.** [Statistical SP](./Statistical_Signal_Processing.ipynb) assumed wide-sense stationarity: statistics that do not depend on absolute time. Now draw a BPSK waveform and ask for its instantaneous power as a function of time. It *pulses* at the symbol rate — the second moment is periodic in $t$, so the signal is flatly not WSS. Students usually treat WSS as a mild technical condition; the point here is that essentially every communication signal violates it, and violating it turns out to be a gift rather than a problem.

**Then the reframe.** Statistics that are periodic can be Fourier-expanded. Expanding the time-varying autocorrelation gives coefficients $R_x^\alpha(\tau)$ indexed by *cycle frequency* $\alpha$ — a second frequency axis, describing the rhythm of the statistics rather than the rhythm of the waveform. Say that twice; conflating $\alpha$ with ordinary frequency is the standard confusion in this workshop.

**The asymmetry is the whole point — state it before the demo.** White noise is genuinely stationary, so its cyclic content at any $\alpha \neq 0$ is **exactly zero** in expectation. A modulated signal's is not. That is not a difference of degree but of kind, and it is what Session 2 cashes in: at $\alpha \neq 0$ noise contributes only *estimation* error, which shrinks with record length, while signal contributes a fixed amount. Ask what that implies for a detector given unlimited time — arbitrarily deep detection below the noise floor, in principle.

**Ask the room where the spikes will be.** Before running: "the symbol rate is 250 Hz and the carrier is 1000 Hz. Which $\alpha$ light up?" The symbol rate (and its multiples) because the symbol timing is periodic; and $2f_c$ because $\cos^2$ contains a DC term and a term at twice the carrier. That second one is worth deriving — $\cos^2\theta = \tfrac12(1 + \cos 2\theta)$ — since it explains why the carrier appears at *twice* its frequency rather than at $f_c$.

**Then name the application.** Those two spikes are a *fingerprint*: they reveal the symbol rate and the carrier frequency of a signal you have not demodulated and whose parameters you were never told. That is blind modulation classification and spectrum sensing, and it is what makes this workshop matter to the [SDR](../Intro_SDR/Software_Defined_Radio.ipynb) track.

**Pacing note.** `cyclic_acf` is a plain Python loop over lags, evaluated at ~100 candidate $\alpha$ — it is slow by design rather than optimised. Real implementations compute the whole spectral correlation surface via FFT methods (the FAM or SSCA algorithms). Say so if anyone asks why a production sensor could do this in real time.
</details>

## 2. Periodic Statistics

💡 **Intuition.** A BPSK signal's *mean power* pulses at the symbol rate — the signal is not WSS, it's **cyclostationary**: statistics periodic in time. Fourier-expand the time-varying autocorrelation and you get the **cyclic autocorrelation** $R_x^\alpha(\tau)$ at *cycle frequencies* α — nonzero exactly at multiples of the symbol rate (and around twice the carrier). White noise, being genuinely stationary, has **zero** cyclic content at any α ≠ 0 — and that asymmetry is an exploitable superpower.

In [2]:
# a BPSK signal, its cyclic autocorrelation at candidate cycle frequencies
fs, sym_rate, fc = 8000, 250, 1000
sps = fs // sym_rate
n_sym = 400
symbols = rng.choice([-1, 1], n_sym)
base = np.repeat(symbols, sps)
t = np.arange(len(base)) / fs
x_clean = base * np.cos(2*np.pi*fc*t)

def cyclic_acf(x, alpha, fs, max_lag=40):
    n_ax = np.arange(len(x))
    rot = x * np.exp(-1j*2*np.pi*alpha*n_ax/fs)
    return np.array([np.mean(rot[k:] * np.conj(x[:len(x)-k])) for k in range(max_lag)])

alphas = np.arange(0, 1200, 12.5)
strength = [np.abs(cyclic_acf(x_clean, a, fs)).max() for a in alphas]
plt.figure(figsize=(8.5, 2.8))
plt.stem(alphas, strength, basefmt=" ", markerfmt=".")
for a_true, name in [(sym_rate, "symbol rate"), (2*fc, "2×carrier")]:
    plt.axvline(a_true, color="r", linestyle=":", linewidth=1)
    plt.text(a_true, max(strength)*0.9, name, fontsize=7, color="r", rotation=90)
plt.xlabel("cycle frequency α [Hz]"); plt.ylabel("max |R_x^α(τ)|")
plt.title("cyclic signature of BPSK: spikes exactly at the symbol rate and 2f_c")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2988402/821798061.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The cyclic autocorrelation is essentially flat across candidate cycle frequencies except for spikes at two places: **250 Hz**, the symbol rate, and **2000 Hz**, twice the carrier. Both were marked in advance, and both are exactly where the theory says they must be.

**Why those two, specifically.** The symbol rate appears because the symbol timing is periodic — the signal's power pulses once per symbol, so its second-order statistics repeat at 250 Hz. Twice the carrier appears from the trigonometry: squaring a carrier gives $\cos^2(2\pi f_c t) = \tfrac12[1 + \cos(4\pi f_c t)]$, so a second-order statistic of a signal carried at $f_c$ contains a component at $2f_c$, never at $f_c$ itself. Students consistently expect a spike at 1000 Hz; the factor of two is a direct consequence of looking at a *quadratic* statistic.

**Note carefully what axis this is.** The horizontal axis is $\alpha$, the **cycle frequency** — the rate at which the signal's statistics repeat — not ordinary frequency. This is a second frequency axis describing the rhythm of the statistics rather than of the waveform. Conflating the two is the standard confusion in this subject; the PSD of this signal would show a hump around 1000 Hz and nothing whatsoever at 250 Hz.

**And here is the asymmetry that makes all of this useful.** White noise is genuinely stationary — its statistics do not vary with time at all — so its cyclic content at every $\alpha \neq 0$ is **exactly zero in expectation**. Not small: zero. A finite record produces only estimation noise, which shrinks as $1/\sqrt{N}$ with record length.

That is a difference in kind rather than degree, and it changes what is possible. An energy detector compares signal power against noise power, so it fails once the signal is weaker than the noise. A cyclic detector evaluates a quantity to which noise contributes *nothing* on average, so with enough record length the signal's rhythm can be pulled out from arbitrarily far below the noise floor. Session 2 measures exactly that.

**One practical framing.** These two spikes constitute a *fingerprint*: they reveal the symbol rate and carrier frequency of a signal nobody demodulated and whose parameters were never supplied. Blind modulation classification and spectrum sensing are built on reading this plot. Note also that `cyclic_acf` here is a deliberately literal Python loop; production sensors compute the whole spectral correlation surface with FFT-based algorithms (FAM, SSCA) fast enough to run live.

---
### 🕐 Session 2 of 3 — *Detection Below the Noise Floor* (~40 min)
**Goal:** energy detection dies at 0 dB; the cyclic detector keeps working well below it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (higher-order statistics).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Detection Below the Noise Floor</b></summary>

**Timing (~40 min).** 10 min why energy detection fails · 8 min what the cyclic detector asks instead · 12 min the demo · 10 min the honest caveats.

**Board first — kill the energy detector properly.** An energy detector asks "is total power above the noise baseline?" Two failures, and the second is the one students miss. It obviously struggles below 0 dB SNR. But worse, it needs to *know* the noise baseline, and in a real receiver that drifts with temperature, gain, and interference. If your noise estimate is uncertain by 1 dB and the signal contributes less than 1 dB of power, no amount of integration time helps — this is the **SNR wall**, and it is a hard limit rather than a slow degradation. Point at `level_uncert_db=1.0` in the code: the demo deliberately models it.

**Then the contrast.** The cyclic detector asks a different question entirely: "is there energy at $\alpha$ = the symbol rate?" Noise contributes *nothing* there in expectation, so there is no baseline to know and nothing to calibrate. Its error comes only from finite record length and shrinks as $1/\sqrt{N}$. This is why cyclic detection has no SNR wall: given enough time, arbitrarily weak signals are detectable in principle.

**Ask the room before running.** "At −10 dB SNR the signal is a tenth the power of the noise. Can anything detect it?" Most say no. Then reveal 8.8σ. The reframe is that detectability depends on what *question* you ask of the data, not only on how much signal there is.

**Read the table as trends, not as precise figures — say this explicitly.** Energy separation collapses 5.7σ → 1.4σ → 0.8σ while the cyclic detector holds 42.5σ → 24.2σ → 8.8σ. The *shape* is the result. The individual numbers come from 20 trials per condition, so a value like 42.5σ is not meaningfully estimated — with 20 samples you cannot resolve separations much beyond a handful of sigma, and a separation statistic with variance in its denominator is noisy at that scale. Treat large values as "far apart" rather than as measurements. If a student asks how to do better, the answer is more trials and an ROC curve rather than a single separation number.

**Do not overclaim.** Three honest limits worth stating: (1) the detector must be told which $\alpha$ to look at — you need to know or search over the symbol rate, and searching costs both computation and false alarms; (2) performance depends on record length, and 60000 samples is 240 symbols, so the comparison is at a fixed and fairly generous budget; (3) real channels add frequency offset and multipath, which smear the cyclic features and degrade this considerably. The demo shows a mechanism under favourable conditions.

**Close on the application.** This is how spectrum sensors detect whether a band is occupied without demodulating anything, and how cognitive radio decides a channel is free. The signal is below the noise floor and its *rhythm* is still visible.
</details>

## 3. The Superpower, Cashed

💡 **Intuition.** An energy detector asks 'is total power above the noise baseline?' — hopeless when SNR < 0 dB and the noise level is uncertain. The **cyclic detector** asks 'is there energy at cycle frequency α = symbol rate?' — and since noise contributes *nothing* there (only estimation noise, shrinking with record length), the signal's rhythm shines through arbitrarily far below the floor, given time. This is how spectrum sensors find hidden transmitters.

In [3]:
def detect_trial(snr_db, with_signal, N_rec=60000, alpha=sym_rate, level_uncert_db=1.0):
    n_sym_r = N_rec // sps
    xb = np.repeat(rng.choice([-1, 1], n_sym_r), sps).astype(float)
    tt = np.arange(len(xb)) / fs
    s = xb * np.cos(2*np.pi*fc*tt)                        # signal power = 1/2
    # REALISM: the receiver's noise level is only known to ±1 dB (temperature, gain drift)
    wobble = 10**(rng.uniform(-level_uncert_db, level_uncert_db)/20)
    noise_sigma = np.sqrt(0.5) * 10**(-snr_db/20) * wobble
    y = (s if with_signal else np.zeros(len(xb))) + noise_sigma * rng.standard_normal(len(xb))
    energy = np.mean(y**2)
    cyc = np.abs(cyclic_acf(y, alpha, fs, max_lag=20)).max()
    return energy, cyc

for snr in [0, -5, -10]:
    e1 = [detect_trial(snr, True)[0] for _ in range(20)]
    e0 = [detect_trial(snr, False)[0] for _ in range(20)]
    c1 = [detect_trial(snr, True)[1] for _ in range(20)]
    c0 = [detect_trial(snr, False)[1] for _ in range(20)]
    def sep(a, b): return (np.mean(a)-np.mean(b)) / np.sqrt(np.var(a)+np.var(b)+1e-18)
    print(f"SNR {snr:+3d} dB:  energy-detector separation {sep(e1,e0):5.1f}σ   cyclic-detector {sep(c1,c0):5.1f}σ")

SNR  +0 dB:  energy-detector separation   5.7σ   cyclic-detector  42.5σ


SNR  -5 dB:  energy-detector separation   1.4σ   cyclic-detector  24.2σ


SNR -10 dB:  energy-detector separation   0.8σ   cyclic-detector   8.8σ


Read the table: the energy detector's separation collapses with SNR (and would collapse entirely under noise-level uncertainty), while the cyclic statistic keeps the classes many σ apart — the rhythm survives where the power story drowns.

---
### 🕐 Session 3 of 3 — *Higher-Order Statistics* (~35 min)
**Goal:** what covariance can't see: kurtosis, the bispectrum, and Gaussian blindness.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Higher-Order Statistics</b></summary>

**Timing (~35 min).** 10 min what covariance cannot see · 8 min cumulants and Gaussian blindness · 12 min the bicoherence demo · 5 min applications.

**Board first — draw the limitation.** A covariance matrix describes an *ellipse*: it captures the spread and the pairwise linear relationships and nothing else. Sketch two very different point clouds with identical covariance — a Gaussian blob and a ring, say. Second-order statistics cannot tell them apart. That is the gap higher-order statistics fill, and it is worth making visual before any formula appears.

**State the master fact and let it do the work.** **All cumulants above second order of a Gaussian are exactly zero.** Not small — zero. Two consequences worth drawing out separately. First, higher-order statistics are automatically blind to Gaussian noise, so they suppress it for free without knowing its level. Second, any nonzero value up there is *provably* non-Gaussian structure. That is the same "difference in kind rather than degree" that made Session 2's cyclic detector work, arriving by a different route, and pointing out the parallel is what unifies the workshop.

**Set up the demo as a magic trick.** Both signals contain tones at $f_1$, $f_2$, and $f_1+f_2$ with *identical amplitudes*. Their power spectra are indistinguishable — the PSD discards phase entirely, so it literally cannot express the difference. Ask the room what could possibly distinguish them. The answer is that in the coupled signal the third tone's phase is *slaved* to $\phi_1 + \phi_2$ at every instant, while in the uncoupled one it drifts independently. Phase *relations* between different frequencies are exactly what second-order statistics throw away.

**Why this matters physically.** Quadratic phase coupling is the fingerprint of a **nonlinearity**. Pass two tones through any squaring element and you get a component at $f_1+f_2$ whose phase is necessarily the sum of the parents' phases — that is what multiplication does. So detecting phase coupling is detecting that a nonlinear process generated the data. Applications: identifying nonlinear distortion in amplifiers, coupled oscillations in EEG, wave–wave interactions in plasma and ocean physics.

**Point at the 0.110 rather than glossing it.** The uncoupled bicoherence is not zero, and students will ask. With `nseg=64` averaging segments, the expected bias floor for genuinely uncorrelated phases is about $1/\sqrt{64} = 0.125$ — so 0.110 is *at the floor*, statistically indistinguishable from zero. That is the honest reading, and it is a much better answer than "small enough." Increase `nseg` and the floor drops; this is worth demonstrating if time allows.

**Ask the room.** "Kurtosis is the fourth cumulant. Where have we used it?" [ICA](./ICA_Blind_Source_Separation.ipynb) steers by it — non-Gaussianity is the compass for finding independent components, and that only works because Gaussians have zero excess kurtosis. If both workshops are on the syllabus, this is the sentence that connects them.

**Honest caveat.** Higher-order statistics are variance-hungry: estimating a third- or fourth-order quantity well takes far more data than estimating a covariance, and they are correspondingly sensitive to outliers. They are the right tool when the structure you need is genuinely invisible at second order, not a free upgrade.
</details>

## 4. Beyond Second Order

💡 **Intuition.** Covariance sees only the ellipse of a distribution. **Higher-order cumulants** see shape: kurtosis (4th) measures tail-heaviness — the compass [ICA](./ICA_Blind_Source_Separation.ipynb) steers by — and the **bispectrum** (the 2-D Fourier transform of the 3rd-order cumulant) detects *phase coupling*: components at $f_1, f_2, f_1{+}f_2$ with locked phases, the fingerprint of nonlinearity. The master fact: **all cumulants above 2nd order of a Gaussian are exactly zero** — so anything nonzero up there is, provably, structure.

In [4]:
# quadratic phase coupling: visible to the bispectrum, invisible to the PSD
N = 2**14
tt = np.arange(N)/fs
f1, f2 = 480, 700
# phases drift slowly and independently between oscillators (physical: separate sources);
# in the COUPLED signal the third tone's phase is SLAVED to ph1+ph2 at every instant
def drifting_phase(N, rate=0.05):
    return np.cumsum(rate * rng.standard_normal(N))
ph1, ph2, ph3 = drifting_phase(N), drifting_phase(N), drifting_phase(N)
coupled   = np.cos(2*np.pi*f1*tt+ph1) + np.cos(2*np.pi*f2*tt+ph2) + 0.7*np.cos(2*np.pi*(f1+f2)*tt + ph1+ph2)
uncoupled = np.cos(2*np.pi*f1*tt+ph1) + np.cos(2*np.pi*f2*tt+ph2) + 0.7*np.cos(2*np.pi*(f1+f2)*tt + ph3)
coupled += 0.3*rng.standard_normal(N); uncoupled += 0.3*rng.standard_normal(N)

def bicoherence_peak(x, f1, f2, fs, nseg=64):
    seg = len(x)//nseg
    num = 0; d1 = 0; d2 = 0
    for k in range(nseg):
        X = np.fft.rfft(x[k*seg:(k+1)*seg] * np.hanning(seg))
        freqs = np.fft.rfftfreq(seg, 1/fs)
        i1, i2 = np.argmin(np.abs(freqs-f1)), np.argmin(np.abs(freqs-f2))
        i12 = np.argmin(np.abs(freqs-(f1+f2)))
        num += X[i1]*X[i2]*np.conj(X[i12])
        d1 += np.abs(X[i1]*X[i2])**2; d2 += np.abs(X[i12])**2
    return np.abs(num) / np.sqrt(d1*d2)

print("PSD at f1, f2, f1+f2 is IDENTICAL for both signals (same magnitudes; only phase RELATIONS differ)")
print(f"bicoherence at (f1, f2):  coupled {bicoherence_peak(coupled, f1, f2, fs):.3f}   "
      f"uncoupled {bicoherence_peak(uncoupled, f1, f2, fs):.3f}")
print("→ near 1 vs near 0: third-order statistics detect the phase LOCK the PSD cannot express")

PSD at f1, f2, f1+f2 is IDENTICAL for both signals (same magnitudes; only phase RELATIONS differ)
bicoherence at (f1, f2):  coupled 0.977   uncoupled 0.110
→ near 1 vs near 0: third-order statistics detect the phase LOCK the PSD cannot express


**What just happened.** Bicoherence **0.977** for the coupled signal against **0.110** for the uncoupled one — from two signals whose power spectra are *identical*.

That identity is the point, so be precise about it. Both signals contain tones at $f_1 = 480$, $f_2 = 700$, and $f_1 + f_2 = 1180$ Hz with the same amplitudes. The PSD measures $|X(f)|^2$ and therefore discards phase entirely; it cannot express the difference between these two signals even in principle. What differs is a phase *relation*: in the coupled signal the third tone's phase is slaved to $\phi_1 + \phi_2$ at every instant, while in the uncoupled one it drifts independently. Second-order statistics are structurally blind to relations between different frequencies.

The bicoherence sees it because it averages the *triple product* $X(f_1)X(f_2)X^*(f_1{+}f_2)$ across segments. When the phases are locked, that product has a consistent phase in every segment and the terms add coherently — near 1. When the third phase drifts freely, the product's phase is random per segment and the sum cancels — near 0.

**Read the 0.110 correctly, because it is not zero.** With `nseg = 64` segments averaged, the expected bias floor for genuinely uncorrelated phases is roughly $1/\sqrt{64} = 0.125$. The measured 0.110 sits right at that floor, so it is **statistically indistinguishable from zero** — random phases cancel only as fast as $1/\sqrt{N}$, never exactly. This is the honest reading, and it matters: quoting 0.110 as "small" invites the question of how small is small, whereas quoting it against its own noise floor answers it. Increase `nseg` and the floor drops accordingly.

**Why this is worth caring about physically.** Quadratic phase coupling is the signature of a **nonlinearity**. Put two tones through any squaring element and the cross-term appears at $f_1 + f_2$ with phase necessarily equal to $\phi_1 + \phi_2$ — that is simply what multiplication does to phases. So detecting phase coupling is detecting that a nonlinear process generated the data, rather than three unrelated oscillators that happen to sit at arithmetically related frequencies. That distinction is invisible to the PSD and is exactly what you need when hunting distortion in an amplifier, coupled rhythms in EEG, or wave interactions in a plasma.

**The unifying fact behind the whole session.** All cumulants above second order of a Gaussian are **exactly zero**. So higher-order statistics ignore Gaussian noise for free, without needing to know its level, and anything nonzero up there is provably non-Gaussian structure. That is the same difference-in-kind that made Session 2's cyclic detector immune to the noise floor, reached by a different route — and it is why [ICA](./ICA_Blind_Source_Separation.ipynb) can use kurtosis as a compass for finding independent components.

**The cost, stated plainly.** Higher-order estimates are variance-hungry — a third- or fourth-order quantity needs far more data than a covariance for comparable precision — and they are sensitive to outliers. Reach for them when the structure you need is genuinely invisible at second order, which here it provably was.

## 5. Conclusion

Modulation puts rhythm into statistics; that rhythm detects signals the PSD loses (σ-separations measured); and third/fourth-order cumulants see phase coupling and non-Gaussian shape where covariance sees nothing at all. When the standard assumptions fail, these are the tools that notice.

---
## Where next

- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — cyclic detection on live captures.
- [ICA](./ICA_Blind_Source_Separation.ipynb) — kurtosis as a steering wheel.
- [Digital Communications](./Digital_Communications.ipynb) — the signals whose rhythms we exploited.